<a href="https://colab.research.google.com/github/safakatakancelik/portfolio-public/blob/main/notebooks/building_attention_from_scratch/attention_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Implemention Attention

1. Raw attention with real word embeddings, no Q/K/V
2. Add W_q, W_k, W_v projection matrices
3. Causal masking (set future positions to −∞ before softmax)
4. Simulate training (parallel, masked) vs inference (sequential, KV cache)
5. Multi-head attention (split dims across heads)
6. Compare & diff attention matrices across variations




---

https://machinelearningmastery.com/the-attention-mechanism-from-scratch/

In [49]:
!pip install torch
!pip install torchtext

In [41]:
import numpy as np
from scipy.special import softmax

In [42]:
np.random.seed(42)

# encoder representations of four different words
word_1 = np.array([1, 0, 0])
word_2 = np.array([0, 1, 0])
word_3 = np.array([1, 1, 0])
word_4 = np.array([0, 0, 1])

# stacking the word embeddings into a single array
words = np.array([word_1, word_2, word_3, word_4])

# generating the weight matrices
W_Q = np.random.randint(3, size=(3, 3))
W_K = np.random.randint(3, size=(3, 3))
W_V = np.random.randint(3, size=(3, 3))

# generating the queries, keys and values
Q = words @ W_Q
K = words @ W_K
V = words @ W_V

# scoring the query vectors against all key vectors
scores = Q @ K.transpose()

# computing the weights by a softmax operation
weights = softmax(scores / K.shape[1] ** 0.5, axis=1)

# computing the attention by a weighted sum of the value vectors
attention = weights @ V

print(attention)

[[0.98522025 1.74174051 0.75652026]
 [0.90965265 1.40965265 0.5       ]
 [0.99851226 1.75849334 0.75998108]
 [0.99560386 1.90407309 0.90846923]]


---

### Step 1: Raw Attention

In [52]:
english_words = ["the", "cat", "sat", "on", "the", "mat"]

In [57]:
# building the vocabulary with random vectors
dim = 8
unique_words = set(english_words)
vocab = {w: np.random.randn(dim) for w in unique_words}

In [60]:
len(vocab)

5

In [61]:
X = np.array([vocab[w] for w in english_words])  # (6, 8)
d_k = X.shape[1]

scores = X @ X.T / np.sqrt(d_k)  # (6, 6)

print("Tokens:", english_words)
print("\nScores shape:", scores.shape)
print("\nRaw attention scores:")
print(np.round(scores, 2))

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']

Scores shape: (6, 6)

Raw attention scores:
[[ 0.35  0.11 -0.1   0.17  0.35  0.14]
 [ 0.11  0.35 -0.07 -0.2   0.11 -0.03]
 [-0.1  -0.07  0.35 -0.09 -0.1  -0.27]
 [ 0.17 -0.2  -0.09  0.35  0.17  0.23]
 [ 0.35  0.11 -0.1   0.17  0.35  0.14]
 [ 0.14 -0.03 -0.27  0.23  0.14  0.35]]
